In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, floor, months_between, get
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.dim_product")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "dim_product"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_product.ipynb"
_silver_table = "silver.product"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.dim_product


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 3
+----------+--------------+-----------------------+------+-----------+------------+----------+--------+------+-----------+------+-----+------+------+------+--------+--------------+-----------+-----------------+---------------+----------+---------------+--------------------------+--------------------+--------------------------+--------------------+
|product_id|product_name  |product_description    |brand |category   |sub_category|department|model   |weight|weight_unit|length|width|height|size  |color |material|product_status|launch_date|discontinued_date|warranty_period|is_deleted|source_file    |created_on                |created_by          |modified_on               |modified_by         |
+----------+--------------+-----------------------+------+-----------+------------+----------+--------+------+-----------+------+-----+------+------+------+--------+--------------+-----------+-----------------+---------------+----------+---------------+--------

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 3


In [10]:
#Adding audit columns

df_silver = df.withColumn("created_on", current_timestamp()) \
              .withColumn("created_by", lit(_script_name)) \
              .withColumn("modified_on", current_timestamp()) \
              .withColumn("modified_by", lit(_script_name))

print(f"SPARK-APP: Silver Table Data count : {df_silver.count()}")

df_silver.show(truncate = False)
df_silver.printSchema()

SPARK-APP: Silver Table Data count : 3
+----------+--------------+-----------------------+------+-----------+------------+----------+--------+------+-----------+------+-----+------+------+------+--------+--------------+-----------+-----------------+---------------+----------+---------------+--------------------------+------------------+--------------------------+------------------+
|product_id|product_name  |product_description    |brand |category   |sub_category|department|model   |weight|weight_unit|length|width|height|size  |color |material|product_status|launch_date|discontinued_date|warranty_period|is_deleted|source_file    |created_on                |created_by        |modified_on               |modified_by       |
+----------+--------------+-----------------------+------+-----------+------------+----------+--------+------+-----------+------+-----+------+------+------+--------+--------------+-----------+-----------------+---------------+----------+---------------+----------------

In [11]:
# Truncate table for full load
dt_dim_product = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_dim_product.delete()
    dt_dim_product.vacuum(0)



In [12]:
# SCD1
dt_dim_product.alias("t").merge(
    df_silver.alias("s"),
    "t.product_id = s.product_id"
).whenMatchedUpdate(set = {
    "product_name":"s.product_name",
    "product_description":"s.product_description",
    "brand":"s.brand",
    "category":"s.category",
    "sub_category":"s.sub_category",
    "department":"s.department",
    "model":"s.model",
    "weight":"s.weight",
    "weight_unit":"s.weight_unit",
    "length":"s.length",
    "width":"s.width",
    "height":"s.height",
    "size":"s.size",
    "color":"s.color",
    "material":"s.material",
    "product_status":"s.product_status",
    "launch_date":"s.launch_date",
    "discontinued_date":"s.discontinued_date",
    "warranty_period":"s.warranty_period",
    "is_deleted":"s.is_deleted",
    "source_file" : "s.source_file",
    "modified_on" : "s.modified_on",
    "modified_by" : "s.modified_by"
    }).whenNotMatchedInsertAll().execute()
                     
print("SPARK-APP: Merge completed")    

SPARK-APP: Merge completed


In [13]:
hist = dt_dim_product.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           3394|                    3|                   0|            3|
+-------+---------------+---------------------+--------------------+-------------+



In [14]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [15]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+-----------+----------+----------+--------------------+-----------+------------+--------------------+------------------+--------------------+------------------+
|layer|schema_name| table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|        created_by|         modified_on|       modified_by|
+-----+-----------+-----------+----------+----------+--------------------+-----------+------------+--------------------+------------------+--------------------+------------------+
| gold|       gold|dim_product|delta-load|     merge|2026-01-29 02:44:...|    success|           3|2026-01-29 02:44:...|gold_product.ipynb|2026-01-29 02:44:...|gold_product.ipynb|
+-----+-----------+-----------+----------+----------+--------------------+-----------+------------+--------------------+------------------+--------------------+------------------+



In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+----------+--------------+--------------------+------+-----------+------------+----------+--------+------+-----------+------+-----+------+------+------+--------+--------------+-----------+-----------------+---------------+----------+--------------------+------------------+--------------------+------------------+---------------+
|product_id|  product_name| product_description| brand|   category|sub_category|department|   model|weight|weight_unit|length|width|height|  size| color|material|product_status|launch_date|discontinued_date|warranty_period|is_deleted|          created_on|        created_by|         modified_on|       modified_by|    source_file|
+----------+--------------+--------------------+------+-----------+------------+----------+--------+------+-----------+------+-----+------+------+------+--------+--------------+-----------+-----------------+---------------+----------+--------------------+------------------+--------------------+------------------+---------------+
|     P

In [17]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [18]:
spark.stop()